## Extended thinking

Extended thinking is Claude's advanced reasoning feature that gives the model time to work through complex problems before generating a final response. Think of it as Claude's "scratch paper" - you can see the reasoning process that leads to the answer, which helps with transparency and often results in better quality responses.

### How Extended Thinking Works

When extended thinking is enabled, Claude's response changes from a simple text block to a structured response containing two parts:

With thinking enabled, you get both the reasoning process and the final answer:


The key benefits include:


* Better reasoning capabilities for complex tasks
* Increased accuracy on difficult problems
* Transparency into Claude's thought process

However, there are important trade-offs:


* Higher costs (you pay for thinking tokens)
* Increased latency (thinking takes time)
* More complex response handling in your code

In [2]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-5-5"

In [3]:
# Helper functions
from anthropic.types import Message

# Magic string to trigger redacted thinking
thinking_test_str = "ANTHROPIC_MAGIC_STRING_TRIGGER_REDACTED_THINKING_46C9A13E193C177646C7398A98432ECCCE4C1253D5E2D82641AC0E52CC2876CB"


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    if thinking:
        # Sonnet 5.5: budget_tokens is no longer accepted (400). Use adaptive
        # thinking; "display": "summarized" is needed to get readable thinking
        # text (the default, "omitted", returns an empty thinking string).
        params["thinking"] = {"type": "adaptive", "display": "summarized"}

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [4]:
messages = []

add_user_message(
    messages,
    "Write a one paragraph guide to recursion."
)

chat(messages, thinking=True)

Message(id='msg_011CgyT79umzJo3UDCJKZxmV', container=None, content=[TextBlock(citations=None, text='Recursion is a technique in which a function solves a problem by calling itself on a smaller version of that same problem. Every recursive solution needs two essential parts: a **base case**, which is the simplest version of the problem that can be answered directly without further calls (such as `factorial(0) = 1`), and a **recursive case**, which breaks the problem into a smaller piece and combines the result with some extra work (such as `factorial(n) = n * factorial(n - 1)`). To write a recursive function, first identify the base case so the process knows when to stop, then ask yourself, "If I could already solve a slightly smaller version of this problem, how would I use that answer to solve the current one?" Trust that the smaller call works correctly, and make sure each call moves closer to the base case; otherwise you\'ll get infinite recursion and a stack overflow. Recursion shi

In [5]:
messages = []

add_user_message(
    messages,
    thinking_test_str
)

chat(messages, thinking=True)

Message(id='msg_011CgyT7U9PVy68qN94x6mEZ', container=None, content=[RedactedThinkingBlock(data='CAQSmAcKEAgTGAI4AUIIdGhpbmtpbmcSDIWXxRfwh9M5pnjmThoMQbX4Bgkdj6tUezVsIjDm3dLN1FWRCmifPU6PdtPR2vARCdH+9bu2wfck8oRUcag1vybmc7OZ7W+8zao554QqtQbh+GbqhfJwuCIYGwg42xsYhtIMMCuVKNkqbF0Re332ux3RJgl+aPEh92T8u3G0bHu4GxVdVsVbG5Ig3EaFlo+3C2x9TVGD+UgIy1gU5/aJ8aHNRVLp1nDGioWOQL47Sc5Q+hhDOvXTZK7VwDBcaQjnEdCMUWv3YJFqMQvcryo9RHAGNJW3+2SB7sMiKu63q+X+PTGZSvXzinP0QTMe70cmFFWPf94NY/bZHQVhGc8FoKQaJhmqjJ0bUYbL0LxApe/p+sI5SB8+ZMgcH0VaL1ty9TB4sN6IpQ4JPjLptEiuKnGdnd0tgIvgnTy3c5U+XILDONgz3bHqpNL1jN2mNVuoyVqlTIWEa6j92s2H4M7OEvDHWOQLk3KhEqOv1/jbtorG8JY29K0zHLJ0LN83YST1pSiyjE2soaXvHBX+8NtFoR+CPp3CoI/T7crBNzhiO1thTeEkZjIlC3AY2Mu0MNmL7gAok+kwHBOlgj2CU11uAlFfWzLVYa+Or3lMHiHshTRX+H4sy/f5vAASpKYYgq+3+nLHsgaXykh6R/x5oxPDlep0Py//J9j1WuhitPBfiEfSlo6yQ+7JmgngeLs7aM292dkWKRLOpt6gmkFVDap2S1noDSnCAlr04rPTwnwuqW2V7hBsLQ4dO8cw6IdTECDfSsbKCSFhQuFswBa2VQGFUBXcR3/C1QjBwvG9XLyYDL/SW+3ugrbiBYAHvkFUF8SPO/yVtKSO4B3icDID3EiYCJkZzmy4Nbsc578w16Yq